# dataloader-batching — ex1: wrap a TensorDataset in a DataLoader and iterate batches

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataloader-batching`. Running the final beacon cell reports progress against the `PyTorch: DataLoader batching` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader batching` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-batching`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-batching"
DD_SUBTOPIC = "PyTorch: DataLoader batching"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `DataLoader(dataset, batch_size, shuffle)` — quick refresher

A `Dataset` yields ONE example at a time. A `DataLoader` wraps a dataset and yields BATCHES — stacking individual `__getitem__` returns into a leading batch dimension.

```
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = DataLoader(testset,  batch_size=64, shuffle=False)
for xb, yb in trainloader:
    ...  # xb: (B, *features), yb: (B, *labels)
```

**Two non-negotiable conventions.** `shuffle=True` for training (so consecutive batches see different examples — required for SGD's stochasticity). `shuffle=False` for validation/test (so metrics are deterministic across runs).

**Partial last batch.** If `len(dataset) % batch_size != 0`, the final batch is smaller. Setting `drop_last=True` discards it (useful when fixed-size batches matter — e.g. BatchNorm with very small batches).

### Exercise 1 — wrap a TensorDataset in a DataLoader and iterate batches

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `DataLoader(dataset, batch_size, shuffle)` to produce stacked-batch iteration with the ARENA-canonical `shuffle=True` for training and `shuffle=False` for eval.
> Keywords: dataloader, batch-size, shuffle
> ```

**KCs targeted:** `dataloader-wraps-dataset`, `dataloader-shuffle-true-for-train-false-for-test`

Implement `ex1_build_loaders(x_train, y_train, x_test, y_test, batch_size)`. The canonical training/eval DataLoader setup.

1. Wrap `(x_train, y_train)` in a `TensorDataset`.
2. Wrap `(x_test, y_test)`  in a `TensorDataset`.
3. Build `train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)`.
4. Build `test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)`.
5. Return `(train_loader, test_loader)`.

Inputs: same-length 2-D feature tensors and 1-D label tensors. `batch_size` may not evenly divide the dataset; let the default `drop_last=False` keep the partial last batch.

The test verifies:
- Batches stack correctly into `(B, *features)` / `(B,)`.
- The training loader yields different batch orderings across two epochs (shuffle on).
- The test loader yields the SAME batch ordering across two epochs (shuffle off — reproducible eval metrics).
- Every example appears in exactly ONE batch per epoch (no duplicates, no drops).

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def ex1_build_loaders(x_train: Tensor, y_train: Tensor,
                      x_test: Tensor,  y_test: Tensor,
                      batch_size: int) -> tuple:
    """Return (train_loader, test_loader). Train shuffles, test does not."""
    raise NotImplementedError()


def _test_ex1():
    # Build a tiny dataset with KNOWN indices encoded into x[:, 0].
    N_train, N_test, F = 20, 6, 3
    x_train = t.stack([t.full((F,), float(i)) for i in range(N_train)])
    y_train = t.arange(N_train)
    x_test  = t.stack([t.full((F,), float(i + 100)) for i in range(N_test)])
    y_test  = t.arange(N_test) + 100
    BATCH = 4

    t.manual_seed(0)
    train_loader, test_loader = ex1_build_loaders(x_train, y_train, x_test, y_test, BATCH)

    # === Train loader: shapes & shuffle ===
    train_batches_epoch1 = [(xb.clone(), yb.clone()) for xb, yb in train_loader]
    # Shape sanity.
    for xb, yb in train_batches_epoch1[:-1]:
        assert xb.shape == (BATCH, F), f'train xb shape {tuple(xb.shape)} != ({BATCH}, {F})'
        assert yb.shape == (BATCH,),    f'train yb shape {tuple(yb.shape)} != ({BATCH},)'
    # Partial last batch.
    last_xb, last_yb = train_batches_epoch1[-1]
    assert last_xb.shape == (N_train % BATCH, F) if (N_train % BATCH) else (BATCH, F), (
        f'partial-last-batch shape unexpected: {tuple(last_xb.shape)}'
    )

    # Every label 0..N_train-1 appears exactly once across all batches.
    all_train_y = t.cat([yb for _, yb in train_batches_epoch1])
    assert all_train_y.shape == (N_train,), f'concat train y wrong size: {tuple(all_train_y.shape)}'
    assert sorted(all_train_y.tolist()) == list(range(N_train)), (
        f'every train index must appear exactly once per epoch; got {sorted(all_train_y.tolist())}'
    )

    # === Train loader: shuffle ===
    train_batches_epoch2 = [(xb.clone(), yb.clone()) for xb, yb in train_loader]
    y1 = t.cat([yb for _, yb in train_batches_epoch1]).tolist()
    y2 = t.cat([yb for _, yb in train_batches_epoch2]).tolist()
    assert y1 != y2, (
        f'train shuffle=True: epoch-1 and epoch-2 orderings must differ; both were {y1}'
    )

    # === Test loader: shuffle=False ===
    test_batches_epoch1 = [(xb.clone(), yb.clone()) for xb, yb in test_loader]
    test_batches_epoch2 = [(xb.clone(), yb.clone()) for xb, yb in test_loader]
    yt1 = t.cat([yb for _, yb in test_batches_epoch1]).tolist()
    yt2 = t.cat([yb for _, yb in test_batches_epoch2]).tolist()
    assert yt1 == yt2, (
        f'test shuffle=False: both epochs must produce identical ordering; got {yt1} vs {yt2}'
    )
    assert yt1 == [100, 101, 102, 103, 104, 105], (
        f'test loader should yield in dataset order; got {yt1}'
    )

    # x and y must be consistently paired (xb[:, 0] == yb for our synthetic dataset).
    for xb, yb in train_batches_epoch1:
        assert t.equal(xb[:, 0].to(t.long), yb), (
            f'(x, y) pairing broken in train batch: x[:,0]={xb[:,0]}, y={yb}; '
            f'did you forget to pair them in a TensorDataset?'
        )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
from torch.utils.data import TensorDataset, DataLoader


def ex1_build_loaders(x_train, y_train, x_test, y_test, batch_size):
    train_ds = TensorDataset(x_train, y_train)
    test_ds  = TensorDataset(x_test,  y_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader
```

**Why `TensorDataset`.** It is the simplest `Dataset` subclass — wraps any number of tensors (all sharing the first-dim length) and `__getitem__(i)` returns the tuple `(x_train[i], y_train[i], ...)`. The `DataLoader` then collates a list of these tuples into stacked batches automatically.

**Why iterating the same loader twice produces different orderings (with shuffle=True).** Each `__iter__` call builds a fresh `RandomSampler` whose permutation is independent. That's what makes successive epochs of training see different batch orderings without you doing anything — and why `shuffle=False` is what you reach for when you want REPRODUCIBLE eval.

**ARENA convention.** Look at 0_2_12 (training loop for feature extraction): the two-line DataLoader pair with `shuffle=True` then `shuffle=False` is identical to what this drill produces.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()